# Notebook 3: Multi-Head Latent Attention (MLA)

## Learning Objectives
- Understand Multi-Head Attention fundamentals
- Learn about KV cache compression in MLA
- Implement MLA from scratch
- Understand memory efficiency gains

## Task 1: Standard Multi-Head Attention

### HINT: MHA Components
```
Q = x @ W_Q,  K = x @ W_K,  V = x @ W_V
attention(Q, K, V) = softmax(Q @ K^T / sqrt(d_k)) @ V
```

Memory for KV cache: `2 * num_heads * seq_len * head_dim`

In [ ]:
import torch
import torch.nn as nn
import math

class MultiHeadAttention(nn.Module):
    """Standard Multi-Head Attention"""
    def __init__(self, hidden_size, num_heads):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_heads = num_heads
        self.head_dim = hidden_size // num_heads
        
        assert hidden_size % num_heads == 0
        
        self.W_q = nn.Linear(hidden_size, hidden_size)
        self.W_k = nn.Linear(hidden_size, hidden_size)
        self.W_v = nn.Linear(hidden_size, hidden_size)
        self.W_o = nn.Linear(hidden_size, hidden_size)
        
    def forward(self, x, causal_mask=True):
        batch_size, seq_len, _ = x.shape
        
        # Project to Q, K, V
        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)
        
        # Reshape for multi-head: [batch, seq, heads, head_dim] -> [batch, heads, seq, head_dim]
        Q = Q.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        
        # Attention scores
        scale = math.sqrt(self.head_dim)
        attn_scores = torch.matmul(Q, K.transpose(-2, -1)) / scale
        
        # Causal mask
        if causal_mask:
            mask = torch.tril(torch.ones(seq_len, seq_len, device=x.device))
            attn_scores = attn_scores.masked_fill(mask == 0, float('-inf'))
        
        # Softmax
        attn_weights = torch.softmax(attn_scores, dim=-1)
        
        # Apply to V
        attn_output = torch.matmul(attn_weights, V)
        
        # Reshape back: [batch, heads, seq, head_dim] -> [batch, seq, hidden]
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, seq_len, self.hidden_size)
        
        return self.W_o(attn_output)

# Test
mha = MultiHeadAttention(hidden_size=512, num_heads=8)
x = torch.randn(2, 16, 512)
output = mha(x)
print(f"Input: {x.shape}")
print(f"Output: {output.shape}")

## Task 2: MLA - KV Cache Compression

### HINT: Key Innovation
MLA compresses K and V into a **latent vector**:

```
Standard MHA:
  K, V: [batch, num_heads, seq_len, head_dim]
  Memory: O(seq_len * num_heads * head_dim)

MLA:
  KV_latent: [batch, latent_dim]
  Memory: O(latent_dim) - constant regardless of seq_len!
```

For 128K context, this is a huge saving!

In [ ]:
class MultiHeadLatentAttention(nn.Module):
    """Multi-Head Latent Attention - compresses KV cache"""
    def __init__(self, hidden_size, num_heads, num_kv_heads=None, latent_dim=None):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_heads = num_heads
        self.num_kv_heads = num_kv_heads or num_heads  # For GQA
        self.head_dim = hidden_size // num_heads
        
        # Latent KV dimension (compressed)
        # Typically much smaller than full KV
        self.latent_dim = latent_dim or (self.num_kv_heads * self.head_dim // 2)
        
        # Q projection (same as standard)
        self.W_q = nn.Linear(hidden_size, hidden_size)
        
        # KV projection to latent space (compressed!)
        self.W_KV = nn.Linear(hidden_size, self.latent_dim * 2)  # Both K and V
        
        # Output projection
        self.W_o = nn.Linear(hidden_size, hidden_size)
        
    def forward(self, x, kv_cache=None, causal_mask=True):
        batch_size, seq_len, _ = x.shape
        
        # Q projection (standard)
        Q = self.W_q(x)
        Q = Q.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        
        # KV projection to latent space
        KV = self.W_KV(x)  # [batch, seq, latent_dim * 2]
        
        # Split into K and V (latent)
        K_latent = KV[..., :self.latent_dim]
        V_latent = KV[..., self.latent_dim:]
        
        # In practice, you'd upcast latent to full dimension for attention
        # This is simplified - shows the compression concept
        K = K_latent.unsqueeze(1).expand(-1, self.num_kv_heads, -1, self.head_dim)
        V = V_latent.unsqueeze(1).expand(-1, self.num_kv_heads, -1, self.head_dim)
        
        # If using GQA, need to expand KV heads to Q heads
        if self.num_kv_heads < self.num_heads:
            # Repeat K, V across Q heads
            repeat_factor = self.num_heads // self.num_kv_heads
            K = K.repeat_interleave(repeat_factor, dim=1)
            V = V.repeat_interleave(repeat_factor, dim=1)
        
        # Attention (same as standard)
        scale = math.sqrt(self.head_dim)
        attn_scores = torch.matmul(Q, K.transpose(-2, -1)) / scale
        
        if causal_mask:
            mask = torch.tril(torch.ones(seq_len, seq_len, device=x.device))
            attn_scores = attn_scores.masked_fill(mask == 0, float('-inf'))
        
        attn_weights = torch.softmax(attn_scores, dim=-1)
        attn_output = torch.matmul(attn_weights, V)
        
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, seq_len, self.hidden_size)
        
        return self.W_o(attn_output)

# Compare memory: MHA vs MLA
seq_len = 512
hidden_size = 512
num_heads = 8
head_dim = hidden_size // num_heads

# MHA KV cache: 2 * heads * seq * head_dim
mha_memory = 2 * num_heads * seq_len * head_dim

# MLA latent: latent_dim (constant!)
latent_dim = num_heads * head_dim // 2  # 50% compression
mla_memory = latent_dim

print(f"MHA KV cache: {mha_memory:,} elements")
print(f"MLA KV cache: {mla_memory:,} elements")
print(f"Memory savings: {(1 - mla_memory/mha_memory)*100:.1f}%")

## Task 3: Grouped Query Attention (GQA)

### HINT: GQA
MLA also works with GQA where there are fewer KV heads than Q heads:
- Fewer KV heads = less memory
- Q heads share KV computation

In [ ]:
# GQA Example: 8 Q heads, 2 KV heads
# This is what DeepSeek V3.2 uses!

class GQA_MLA(nn.Module):
    """MLA with Grouped Query Attention"""
    def __init__(self, hidden_size, num_q_heads, num_kv_heads):
        super().__init__()
        self.num_q_heads = num_q_heads
        self.num_kv_heads = num_kv_heads
        self.head_dim = hidden_size // num_q_heads
        
        # KV latent (shared across fewer KV heads)
        self.latent_dim = num_kv_heads * self.head_dim // 2
        
        self.W_q = nn.Linear(hidden_size, hidden_size)
        self.W_KV = nn.Linear(hidden_size, self.latent_dim * 2)
        self.W_o = nn.Linear(hidden_size, hidden_size)
        
    def forward(self, x):
        batch_size, seq_len, _ = x.shape
        
        Q = self.W_q(x).view(batch_size, seq_len, self.num_q_heads, self.head_dim).transpose(1, 2)
        
        KV = self.W_KV(x)
        K_latent = KV[..., :self.latent_dim]
        V_latent = KV[..., self.latent_dim:]
        
        # Expand to KV heads
        K = K_latent.unsqueeze(1).expand(-1, self.num_kv_heads, -1, self.head_dim)
        V = V_latent.unsqueeze(1).expand(-1, self.num_kv_heads, -1, self.head_dim)
        
        # Expand KV heads to Q heads
        repeat_factor = self.num_q_heads // self.num_kv_heads
        K = K.repeat_interleave(repeat_factor, dim=1)
        V = V.repeat_interleave(repeat_factor, dim=1)
        
        # Attention
        scale = math.sqrt(self.head_dim)
        attn_scores = torch.matmul(Q, K.transpose(-2, -1)) / scale
        mask = torch.tril(torch.ones(seq_len, seq_len, device=x.device))
        attn_scores = attn_scores.masked_fill(mask == 0, float('-inf'))
        attn_weights = torch.softmax(attn_scores, dim=-1)
        attn_output = torch.matmul(attn_weights, V)
        
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, seq_len, self.hidden_size)
        return self.W_o(attn_output)

# DeepSeek V3.2 config: 128 Q heads, 8 KV heads
gqa = GQA_MLA(hidden_size=512, num_q_heads=8, num_kv_heads=2)
x = torch.randn(2, 16, 512)
output = gqa(x)
print(f"GQA config: 8 Q heads, 2 KV heads")
print(f"Input: {x.shape}")
print(f"Output: {output.shape}")

## Verification

Complete these checks:
1. ✅ Understand standard MHA
2. ✅ Can explain KV compression in MLA
3. ✅ Can implement MLA from scratch
4. ✅ Understand GQA + MLA combination